# Esplorativa BDAP - Banca Dati Amministrazioni Pubbliche

Notebook per esplorare e analizzare dati BDAP (MEF/RGS)

## Fonte Dati

**BDAP** (Banca Dati Amministrazioni Pubbliche)
- Ente: MEF/RGS (Ministero Economia e Finanze / Ragioneria Generale dello Stato)
- URL: https://bdap-opendata.rgs.mef.gov.it
- Dataset totali: ~3600
- API: CKAN

## Contenuto Notebook

**SEZIONE 1: SIOPE (Dati Finanziari)**
- Download dataset SIOPE
- Filtro comuni
- Analisi esplorativa completa

**SEZIONE 2: Altri Dataset BDAP (Esplorativa)**
- Esplorazione dataset disponibili
- Test download altri dati (certificati bilancio, patrimonio, personale, ecc.)

---

## Setup

In [ ]:
import requests
import pandas as pd
import json
import time
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Setup cartelle
data_raw = Path('../data/raw')
data_raw.mkdir(parents=True, exist_ok=True)

# Configurazione matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("Setup completato")
print(f"Cartella dati: {data_raw.absolute()}")

---

# SEZIONE 1: SIOPE (Dati Finanziari)

**SIOPE** = Sistema Informativo Operazioni Enti Pubblici

Dati finanziari mensili di tutti gli enti pubblici italiani:
- Entrate e Spese dettagliate
- Granularita mensile
- Codifiche standardizzate
- Formato: CSV (sep=`;`, encoding=`latin-1`)

## 1.1 Funzioni Download SIOPE

In [ ]:
def download_bdap_dataset_completo(dataset_name, anno, regione, tipo, save_path):
    """
    Scarica dataset SIOPE completo usando URL diretto BDAP
    
    Args:
        dataset_name: Nome dataset (es. "SIOPE-Movimenti-cumulati-mensili-di-Entrata")
        anno: Anno dataset (es. 2021, 2025)
        regione: Nome regione (es. "Lazio")
        tipo: "Entrata" o "Spesa"
        save_path: Path cartella destinazione
    
    Returns:
        Path del file scaricato, o None se errore
    """
    
    url = f"https://bdap-opendata.rgs.mef.gov.it/export/csv/{anno}---{regione}---{dataset_name}.csv"
    
    print(f"\nDownload BDAP Dataset")
    print("="*70)
    print(f"Anno: {anno}")
    print(f"Regione: {regione}")
    print(f"Tipo: {tipo}")
    print(f"URL: {url}\n")
    
    try:
        print("Download in corso...")
        response = requests.get(url, timeout=300, stream=True)
        
        if response.status_code == 200:
            filename = f"{anno}_{regione}_{tipo}_completo.csv"
            filepath = Path(save_path) / filename
            
            total_size = 0
            with open(filepath, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
                        total_size += len(chunk)
            
            size_mb = total_size / (1024 * 1024)
            print(f"OK - Download completato")
            print(f"File: {filename}")
            print(f"Dimensione: {size_mb:.2f} MB")
            
            return filepath
            
        else:
            print(f"ERRORE HTTP {response.status_code}")
            return None
            
    except Exception as e:
        print(f"ERRORE: {e}")
        return None


def download_all_siope_datasets(anni, regione="Lazio", save_path=None):
    """
    Scarica dataset SIOPE per piu anni (Entrate + Spese)
    
    Args:
        anni: Lista anni (es. [2021, 2022, 2023])
        regione: Nome regione (default "Lazio")
        save_path: Path cartella destinazione
    
    Returns:
        Lista Path dei file scaricati
    """
    
    if save_path is None:
        save_path = Path('../data/raw/bdap/siope')
    
    save_path.mkdir(parents=True, exist_ok=True)
    datasets = []
    
    for anno in anni:
        # Entrate
        print(f"\n{'='*70}")
        print(f"Download {anno} - Entrate")
        file_e = download_bdap_dataset_completo(
            dataset_name="SIOPE-Movimenti-cumulati-mensili-di-Entrata",
            anno=anno,
            regione=regione,
            tipo="Entrata",
            save_path=save_path
        )
        if file_e:
            datasets.append(file_e)
        
        time.sleep(2)
        
        # Spese
        print(f"\n{'='*70}")
        print(f"Download {anno} - Spese")
        file_s = download_bdap_dataset_completo(
            dataset_name="SIOPE-Movimenti-cumulati-mensili-di-Spesa",
            anno=anno,
            regione=regione,
            tipo="Spesa",
            save_path=save_path
        )
        if file_s:
            datasets.append(file_s)
        
        time.sleep(2)
    
    print(f"\n\n{'='*70}")
    print(f"Download completati: {len(datasets)} dataset")
    
    return datasets


def filter_comuni(input_file, output_file=None):
    """
    Filtra solo comuni da CSV SIOPE completo
    
    Il CSV BDAP contiene tutti gli enti pubblici mischiati.
    Codice tipo ente:
    - CO = Comuni (quello che vogliamo)
    - CM = Comunita Montane
    - PR = Province
    - ecc.
    
    Args:
        input_file: Path CSV completo
        output_file: Path CSV output (default: stesso nome + _SOLO_COMUNI)
    
    Returns:
        DataFrame filtrato
    """
    
    input_path = Path(input_file)
    
    if output_file is None:
        output_file = input_path.parent / input_path.name.replace('_completo', '_SOLO_COMUNI')
    
    print(f"\nFiltro comuni")
    print("="*70)
    print(f"Input: {input_path.name}")
    
    # Carica
    df = pd.read_csv(input_path, sep=';', encoding='latin-1', low_memory=False)
    print(f"Righe totali: {len(df):,}")
    
    # Filtra codice CO (comuni)
    comuni_df = df[df['Codice Tipologia Ente BDAP'] == 'CO'].copy()
    print(f"Righe comuni: {len(comuni_df):,}")
    print(f"Comuni unici: {comuni_df['Descrizione Ente BDAP'].nunique()}")
    
    # Salva
    comuni_df.to_csv(output_file, sep=';', index=False, encoding='latin-1')
    output_size_mb = Path(output_file).stat().st_size / (1024*1024)
    print(f"\nSalvato: {Path(output_file).name}")
    print(f"Dimensione: {output_size_mb:.1f} MB")
    
    return comuni_df

## 1.2 Esecuzione Download

Scarica dati SIOPE e filtra solo comuni

In [ ]:
# Download anno 2025 (Entrate + Spese)
# NOTA: Commenta questa cella se hai gia scaricato i dati

files = download_all_siope_datasets(anni=[2025])

# Per scaricare serie storica:
# files = download_all_siope_datasets(anni=[2021, 2022, 2023, 2024, 2025])

In [ ]:
# Filtra solo comuni da CSV completo
# NOTA: Se hai gia il file _SOLO_COMUNI.csv, salta questa cella

df_comuni_entrate = filter_comuni(
    input_file='../data/raw/bdap/siope/2025_Lazio_Entrata_completo.csv'
)

# Se hai scaricato anche le spese:
# df_comuni_spese = filter_comuni(
#     input_file='../data/raw/bdap/siope/2025_Lazio_Spesa_completo.csv'
# )

## 1.3 Analisi Esplorativa SIOPE

### 1.3.1 Caricamento Dati

In [ ]:
# Carica CSV comuni filtrato
df = pd.read_csv('../data/raw/bdap/siope/2025_Lazio_Entrata_SOLO_COMUNI.csv',
                 sep=';', encoding='latin-1', low_memory=False)

# Converti importo da stringa a float
df['Importo'] = df['Importo cumulato'].astype(str).str.replace(',', '.').astype(float)

print(f"Dataset caricato: {len(df):,} righe")
print(f"Periodo: {df['Anno/Mese calendario'].min()} - {df['Anno/Mese calendario'].max()}")
print(f"\nPrime colonne:")
df.head()

### 1.3.2 Statistiche Base

In [ ]:
# Informazioni generali
print("STATISTICHE GENERALI")
print("="*70)
print(f"Comuni unici: {df['Descrizione Ente BDAP'].nunique()}")
print(f"Mesi disponibili: {df['Anno/Mese calendario'].nunique()}")
print(f"Voci di entrata uniche: {df['Descrizione CG'].nunique()}")

totale = df['Importo'].sum()
print(f"\nImporto totale: EUR {totale:,.2f}")
print(f"Media per comune: EUR {totale/df['Descrizione Ente BDAP'].nunique():,.2f}")

In [ ]:
# Top 10 comuni per entrate totali
top_comuni = df.groupby('Descrizione Ente BDAP')['Importo'].sum().nlargest(10)

print("\nTOP 10 COMUNI PER ENTRATE TOTALI")
print("="*70)
for i, (comune, importo) in enumerate(top_comuni.items(), 1):
    print(f"{i:2d}. {comune:40s} EUR {importo:15,.2f}")

### 1.3.3 Analisi Temporale

In [ ]:
# Entrate mensili aggregate
mensili = df.groupby('Anno/Mese calendario')['Importo'].sum().sort_index()

print("ENTRATE MENSILI AGGREGATE (tutti i comuni)")
print("="*70)
for mese, importo in mensili.items():
    print(f"{mese}: EUR {importo:,.2f}")

# Grafico
fig, ax = plt.subplots(figsize=(12, 6))
mensili.plot(kind='bar', ax=ax)
ax.set_title('Entrate Mensili Aggregate - Comuni Lazio 2025', fontsize=14, fontweight='bold')
ax.set_xlabel('Mese')
ax.set_ylabel('Importo (EUR)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 1.3.4 Analisi per Voce di Bilancio

In [ ]:
# Top 10 voci di entrata (Titoli)
top_voci = df.groupby('Descrizione Titolo CG')['Importo'].sum().nlargest(10)

print("TOP 10 VOCI DI ENTRATA")
print("="*70)
for i, (voce, importo) in enumerate(top_voci.items(), 1):
    pct = (importo / df['Importo'].sum()) * 100
    print(f"{i:2d}. {voce[:50]:50s} EUR {importo:15,.2f} ({pct:5.1f}%)")

# Grafico
fig, ax = plt.subplots(figsize=(12, 8))
top_voci.plot(kind='barh', ax=ax)
ax.set_title('Top 10 Voci di Entrata - Comuni Lazio 2025', fontsize=14, fontweight='bold')
ax.set_xlabel('Importo (EUR)')
plt.tight_layout()
plt.show()

### 1.3.5 Analisi Comune Specifico

In [ ]:
# Esempio: analizza Roma
comune_scelto = 'ROMA CAPITALE'

df_comune = df[df['Descrizione Ente BDAP'] == comune_scelto]

print(f"ANALISI: {comune_scelto}")
print("="*70)
print(f"Righe totali: {len(df_comune):,}")
print(f"Importo totale: EUR {df_comune['Importo'].sum():,.2f}")
print(f"\nTop 5 voci di entrata:")

top5_comune = df_comune.groupby('Descrizione Titolo CG')['Importo'].sum().nlargest(5)
for voce, importo in top5_comune.items():
    print(f"  - {voce[:50]:50s} EUR {importo:,.2f}")

# Evoluzione mensile
mensili_comune = df_comune.groupby('Anno/Mese calendario')['Importo'].sum().sort_index()

fig, ax = plt.subplots(figsize=(12, 6))
mensili_comune.plot(kind='line', marker='o', ax=ax)
ax.set_title(f'Entrate Mensili - {comune_scelto}', fontsize=14, fontweight='bold')
ax.set_xlabel('Mese')
ax.set_ylabel('Importo (EUR)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 1.3.6 Confronto Comuni

In [ ]:
# Confronta top 5 comuni
top5_comuni = df.groupby('Descrizione Ente BDAP')['Importo'].sum().nlargest(5).index

# Pivot: mesi x comuni
pivot = df[df['Descrizione Ente BDAP'].isin(top5_comuni)].pivot_table(
    index='Anno/Mese calendario',
    columns='Descrizione Ente BDAP',
    values='Importo',
    aggfunc='sum'
)

# Grafico
fig, ax = plt.subplots(figsize=(14, 8))
pivot.plot(kind='line', marker='o', ax=ax)
ax.set_title('Confronto Entrate Mensili - Top 5 Comuni', fontsize=14, fontweight='bold')
ax.set_xlabel('Mese')
ax.set_ylabel('Importo (EUR)')
ax.legend(title='Comune', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 1.4 Utility

In [ ]:
def cleanup_siope_folder(siope_dir='../data/raw/bdap/siope', dry_run=True):
    """
    Pulisce file vecchi/test dalla cartella SIOPE
    
    Tiene solo:
    - CSV con '_completo.csv' o '_SOLO_COMUNI.csv'
    - CSV > 10 MB
    - File .json
    
    Args:
        siope_dir: Path cartella SIOPE
        dry_run: Se True, mostra solo cosa verrebbe eliminato
    """
    
    siope_path = Path(siope_dir)
    
    if not siope_path.exists():
        print(f"ERRORE: Directory {siope_dir} non trovata")
        return
    
    all_files = [f for f in siope_path.glob('*') if f.is_file()]
    keep_files = []
    remove_files = []
    
    for f in all_files:
        size_mb = f.stat().st_size / (1024*1024)
        
        keep = False
        if '_completo.csv' in f.name or '_SOLO_COMUNI.csv' in f.name:
            keep = True
        elif f.suffix == '.csv' and size_mb > 10:
            keep = True
        elif f.suffix == '.json':
            keep = True
        
        if keep:
            keep_files.append((f, size_mb))
        else:
            remove_files.append((f, size_mb))
    
    print("\nCLEANUP CARTELLA SIOPE")
    print("="*70)
    print(f"File totali: {len(all_files)}")
    print(f"Da tenere: {len(keep_files)}")
    print(f"Da eliminare: {len(remove_files)}")
    
    if remove_files:
        print("\nFile da eliminare:")
        for f, size in remove_files:
            print(f"  - {f.name} ({size:.1f} MB)")
        
        space_saved = sum(size for _, size in remove_files)
        print(f"\nSpazio recuperabile: {space_saved:.1f} MB")
        
        if dry_run:
            print("\n[DRY RUN] Nessun file eliminato")
            print("Per eliminare: cleanup_siope_folder(dry_run=False)")
        else:
            for f, _ in remove_files:
                f.unlink()
                print(f"Eliminato: {f.name}")
    else:
        print("\nNessun file da eliminare")


# Esegui in dry-run
cleanup_siope_folder(dry_run=True)

---

# SEZIONE 2: Altri Dataset BDAP (Esplorativa)

BDAP contiene ~3600 dataset oltre a SIOPE.

Questa sezione esplora altri dataset utili per analisi comunali:
- Certificati di Bilancio Consuntivo
- Patrimonio e Indebitamento
- Personale e Spese
- Tributi Locali
- Servizi Pubblici

**NOTA:** Codice esplorativo minimale per verificare disponibilita e struttura dati.

## 2.1 Esplora Dataset Disponibili

In [ ]:
# Cerca dataset per keyword
BDAP_BASE = "https://bdap-opendata.rgs.mef.gov.it/SpodCkanApi/api/3/action"

def search_bdap_datasets(keyword, max_results=20):
    """
    Cerca dataset BDAP per keyword
    
    Args:
        keyword: Parola chiave
        max_results: Numero massimo risultati
    
    Returns:
        Lista dataset trovati
    """
    
    try:
        response = requests.get(
            f"{BDAP_BASE}/package_search",
            params={'q': keyword, 'rows': max_results},
            timeout=30
        )
        
        if response.status_code == 200:
            data = response.json()
            if data.get('success'):
                results = data['result']['results']
                count = data['result']['count']
                
                print(f"\nTrovati {count} dataset per '{keyword}' (mostro {len(results)})")
                print("="*70)
                
                for i, ds in enumerate(results, 1):
                    title = ds.get('title', 'N/A')
                    name = ds.get('name', 'N/A')
                    n_resources = len(ds.get('resources', []))
                    
                    print(f"{i:2d}. {title}")
                    print(f"    ID: {name}")
                    print(f"    Risorse: {n_resources}")
                    print()
                
                return results
        
        print(f"ERRORE: Status {response.status_code}")
        return []
        
    except Exception as e:
        print(f"ERRORE: {e}")
        return []


# Test ricerca
# Prova diverse keyword
keywords = [
    'bilancio consuntivo',
    'patrimonio',
    'personale',
    'tributi',
    'debiti'
]

for kw in keywords[:1]:  # Prova prima keyword
    results = search_bdap_datasets(kw, max_results=10)
    time.sleep(1)

## 2.2 Test Download Dataset Specifico

In [ ]:
def get_dataset_info(dataset_id):
    """
    Recupera informazioni dettagliate su un dataset
    
    Args:
        dataset_id: ID o name del dataset
    
    Returns:
        Dizionario con info dataset
    """
    
    try:
        response = requests.get(
            f"{BDAP_BASE}/package_show",
            params={'id': dataset_id},
            timeout=30
        )
        
        if response.status_code == 200:
            data = response.json()
            if data.get('success'):
                ds = data['result']
                
                print(f"\nDETTAGLI DATASET")
                print("="*70)
                print(f"Titolo: {ds.get('title')}")
                print(f"Descrizione: {ds.get('notes', 'N/A')[:200]}...")
                print(f"Ultimo aggiornamento: {ds.get('metadata_modified', 'N/A')[:10]}")
                print(f"\nRisorse disponibili: {len(ds.get('resources', []))}")
                
                for i, res in enumerate(ds.get('resources', []), 1):
                    print(f"\n  {i}. {res.get('name', 'N/A')}")
                    print(f"     Formato: {res.get('format', 'N/A')}")
                    print(f"     URL: {res.get('url', 'N/A')[:80]}...")
                
                return ds
        
        print(f"ERRORE: Status {response.status_code}")
        return None
        
    except Exception as e:
        print(f"ERRORE: {e}")
        return None


# Test: prendi primo risultato dalla ricerca precedente
# Decommentare per testare:
# if results:
#     dataset_info = get_dataset_info(results[0]['id'])

## 2.3 Test Download Dati

Template per scaricare e verificare altri dataset BDAP

In [ ]:
# TEMPLATE per download altri dataset
# Adattare URL e parametri in base al dataset specifico

def test_download_bdap_generic(url, output_file, encoding='latin-1', sep=';'):
    """
    Test download generico dataset BDAP
    
    Args:
        url: URL risorsa
        output_file: Nome file output
        encoding: Encoding CSV (default latin-1)
        sep: Separatore CSV (default ;)
    
    Returns:
        DataFrame se successo, None altrimenti
    """
    
    print(f"\nTest download: {output_file}")
    print("="*70)
    
    try:
        # Download
        response = requests.get(url, timeout=120, stream=True)
        
        if response.status_code == 200:
            filepath = data_raw / 'bdap' / output_file
            
            with open(filepath, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
            
            size_mb = filepath.stat().st_size / (1024*1024)
            print(f"Download OK: {size_mb:.2f} MB")
            
            # Carica e verifica
            df = pd.read_csv(filepath, sep=sep, encoding=encoding, nrows=10)
            print(f"\nPrime righe caricate:")
            print(f"Righe: {len(df)}")
            print(f"Colonne: {len(df.columns)}")
            print(f"\nColonne disponibili:")
            for col in df.columns[:10]:
                print(f"  - {col}")
            
            return df
        
        print(f"ERRORE: Status {response.status_code}")
        return None
        
    except Exception as e:
        print(f"ERRORE: {e}")
        return None


# Esempio utilizzo (decommentare per testare):
# url_test = "https://bdap-opendata.rgs.mef.gov.it/export/csv/..."  # URL specifico
# df_test = test_download_bdap_generic(url_test, 'test_dataset.csv')

---

# NOTE FINALI

## Struttura Dati CSV SIOPE

**Encoding:** `latin-1` (non utf-8)

**Separatore:** `;` (punto e virgola)

**Colonne principali:**
- `Codice istat comune`: Codice ISTAT
- `Codice Tipologia Ente BDAP`: CO=Comuni, CM=Comunita Montane, PR=Province
- `Descrizione Ente BDAP`: Nome ente
- `Anno/Mese calendario`: Formato YYYY/MM
- `Descrizione Titolo CG`: Categoria voce bilancio
- `Descrizione CG`: Descrizione dettagliata
- `Importo cumulato`: Importo (virgola come decimale)
- `Popolazione ISTAT`: Abitanti

---

## TODO

**SIOPE:**
- [ ] Scaricare serie storica 2021-2024
- [ ] Analizzare Spese oltre a Entrate
- [ ] Confronto Entrate vs Spese
- [ ] Analisi per dimensione comune
- [ ] Export report PDF/HTML

**Altri dataset BDAP:**
- [ ] Identificare dataset certificati bilancio
- [ ] Testare download patrimonio
- [ ] Verificare dati personale
- [ ] Esplorare tributi locali

**Infrastruttura:**
- [ ] Database SQLite
- [ ] Pipeline ETL automatica
- [ ] API Flask

---